# Legal Clause Analyst — Gradio App

This notebook loads the fine-tuned Llama 3.1 8B QLoRA model and launches a Gradio app for **full-document analysis**:
- Upload a PDF or TXT contract
- Get a **side-by-side clause-by-clause comparison** of the base Llama 3.1 8B Instruct model vs. the fine-tuned QLoRA model

The base-vs-fine-tuned comparison runs through the same model instance — PEFT's `disable_adapter()` context manager toggles the LoRA layer off so we don't have to load two 8B models into VRAM.

**Requirements:** Google Colab with T4 GPU enabled.

## 1. Install Dependencies

In [ ]:
!pip install unsloth gradio pdfplumber

## 2. Mount Drive & Load Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "/content/drive/MyDrive/legal-clauses/legal-llama3-qlora"
)
FastLanguageModel.for_inference(model)

## 3. Analysis Functions

In [ ]:
import json
import re
import contextlib
import pdfplumber
import pandas as pd


def _generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True
    )


def analyze_clause(clause_text, use_base=False):
    """Send a single clause through the model and return structured JSON.

    When use_base=True, the LoRA adapter is disabled so the call runs
    through the unmodified base Llama 3.1 8B Instruct weights.
    """
    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are a legal clause analyst. Analyze the following legal clause "
        "and provide a structured JSON response with: type, summary, "
        "risk (Low/Medium/High), reason, and suggestion."
        "<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
        f"{clause_text}"
        "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    )

    ctx = model.disable_adapter() if use_base else contextlib.nullcontext()
    with ctx:
        response = _generate(prompt)

    # Strip common markdown fences before parsing
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", response.strip())
    try:
        return json.loads(cleaned)
    except Exception:
        return {
            "type": "unknown",
            "summary": response,
            "risk": "N/A",
            "reason": "",
            "suggestion": "",
        }

In [ ]:
def extract_text_from_pdf(file_path):
    """Extract all text from a PDF using pdfplumber."""
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


def split_into_clauses(text):
    """Split a legal document into individual clauses."""
    pattern = (
        r'\n\s*(?='
        r'(?:\d+[\.\)]\s)'
        r'|(?:Section\s+\d)'
        r'|(?:Article\s+\d)'
        r'|(?:ARTICLE\s+\d)'
        r'|(?:SECTION\s+\d)'
        r'|(?:\([a-z]\)\s)'
        r'|(?:\([ivxlc]+\)\s)'
        r')'
    )
    chunks = re.split(pattern, text)

    clauses = []
    for chunk in chunks:
        chunk = chunk.strip()
        if len(chunk) >= 80:
            clauses.append(chunk)

    if len(clauses) < 2:
        clauses = [
            p.strip() for p in text.split("\n\n") if len(p.strip()) >= 80
        ]

    return clauses


def _risk_breakdown(values):
    counts = pd.Series(values).value_counts().to_dict()
    return " | ".join(f"{risk}: {count}" for risk, count in counts.items())


def analyze_document(file):
    """Upload handler: run every clause through both base and fine-tuned models."""
    if file is None:
        return None, None, "No file uploaded."

    if file.name.endswith(".pdf"):
        text = extract_text_from_pdf(file.name)
    else:
        with open(file.name, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

    if not text.strip():
        return None, None, "Could not extract text from the file."

    clauses = split_into_clauses(text)
    if not clauses:
        return None, None, "No clauses detected in the document."

    base_rows, ft_rows = [], []
    for i, clause in enumerate(clauses):
        snippet = clause[:2000]
        preview = clause[:150] + "..." if len(clause) > 150 else clause

        base = analyze_clause(snippet, use_base=True)
        ft = analyze_clause(snippet, use_base=False)

        base_rows.append({
            "#": i + 1,
            "Clause (preview)": preview,
            "Type": base.get("type", "N/A"),
            "Risk": base.get("risk", "N/A"),
            "Summary": base.get("summary", "N/A"),
            "Suggestion": base.get("suggestion", "N/A"),
        })
        ft_rows.append({
            "#": i + 1,
            "Clause (preview)": preview,
            "Type": ft.get("type", "N/A"),
            "Risk": ft.get("risk", "N/A"),
            "Summary": ft.get("summary", "N/A"),
            "Suggestion": ft.get("suggestion", "N/A"),
        })

    base_df = pd.DataFrame(base_rows)
    ft_df = pd.DataFrame(ft_rows)

    total = len(ft_df)
    summary = (
        f"**{total} clauses analyzed**  \n"
        f"**Base model risk breakdown:** {_risk_breakdown(base_df['Risk'])}  \n"
        f"**Fine-tuned model risk breakdown:** {_risk_breakdown(ft_df['Risk'])}"
    )

    return base_df, ft_df, summary

## 4. Launch Gradio App

In [ ]:
import gradio as gr


with gr.Blocks(title="Legal Clause Analyst") as demo:
    gr.Markdown("# Legal Clause Analyst")
    gr.Markdown(
        "Upload a **PDF** or **TXT** contract. The tool splits it into "
        "clauses and runs each one through **both** the base Llama 3.1 8B "
        "Instruct model and the fine-tuned QLoRA model so you can compare "
        "the outputs row-by-row."
    )

    file_input = gr.File(
        label="Upload Contract",
        file_types=[".pdf", ".txt"],
    )
    doc_btn = gr.Button("Analyze Document", variant="primary")
    risk_summary = gr.Markdown(label="Risk Summary")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Base Model — Llama 3.1 8B Instruct")
            base_output = gr.Dataframe(
                label="Base model analysis",
                wrap=True,
            )
        with gr.Column():
            gr.Markdown("### Fine-tuned — Legal QLoRA")
            ft_output = gr.Dataframe(
                label="Fine-tuned model analysis",
                wrap=True,
            )
    doc_btn.click(
        fn=analyze_document,
        inputs=file_input,
        outputs=[base_output, ft_output, risk_summary],
    )

demo.launch(share=True)